[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C04_AI_Agents_Course/02_react_agent/02_react_from_scratch.ipynb)

# 模块 02 · 从零写 ReAct Agent

**配套讲解**：`02_讲解.html`（建议先读完第 1–3 节再动手；第 5/6 节与本 notebook 的失败分析、Reflexion 实验直接对应）。

**模块定位**：agent 不是框架魔法——本 notebook 用**裸 Python（零 agent 框架）**手写完整的 ReAct 循环：

1. 全离线工具集：`calculator` / `wiki_lookup` / `note_write` / `note_read`；
2. ReAct 文法 system prompt（Thought / Action / Action Input / Observation / Final Answer）+ few-shot；
3. `parse_step` regex 解析器（含格式漂移兜底）；
4. `react_agent` 主循环（`max_steps` 预算 + 完整轨迹记录）；
5. 失败模式触发与自动分类、Reflexion 自我反思重试。

**算力说明**（CPU 即可，零依赖也能跑完全部教学轨迹）：

| 情况 | LLM 后端 |
|---|---|
| 设置了 `OPENAI_API_KEY` | `gpt-4o-mini`（API） |
| 无 key，且手动设 `USE_LOCAL_LLM = True` | `Qwen/Qwen2.5-1.5B-Instruct`（**约 3.1GB 下载**，CPU 可跑，MPS/GPU 更快） |
| 默认 | `ScriptedMockLLM` —— 按预置脚本回放，教学轨迹**零依赖、完全可复现** |

> Mock 后端不是"假装能跑"：解析器、工具执行、循环控制、轨迹记录全部是真实代码，只有"模型脑子"被脚本替代。这正是模块 03 中对 harness 做单元测试的标准做法。

In [ ]:
import os, re, ast, json, operator
from collections import Counter

HAS_OPENAI_KEY = bool(os.environ.get("OPENAI_API_KEY"))
USE_LOCAL_LLM = False   # 改为 True 则使用本地 Qwen2.5-1.5B-Instruct（约 3.1GB 下载）

if HAS_OPENAI_KEY:
    BACKEND = "openai"
elif USE_LOCAL_LLM:
    BACKEND = "local"
else:
    BACKEND = "mock"

print(f"LLM 后端: {BACKEND}")

## 1 · 全离线工具集

四个工具，三条纪律（与讲解第 2/3 节呼应）：

- **绝不使用 `eval`**：`calculator` 用 `ast.parse` + 白名单节点遍历做安全求值——模型输出是不可信文本；
- **永远返回字符串**：工具结果要回填进模型上下文（Observation 行）；
- **失败返回 `ERROR: ...` 而不是抛异常**：错误信息是回喂给模型的修复线索（讲解第 3 节"解析失败 ≠ 轨迹失败"同理）。

`wiki_lookup` 是一个内置 15 条词条的**离线迷你百科**——刻意做小，方便我们一会儿故意问它不知道的问题来触发失败模式。

In [ ]:
# ---------- 工具 1: calculator —— 安全算术求值（禁 eval） ----------
_BINOPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
           ast.Mod: operator.mod, ast.Pow: operator.pow}
_UNARY = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def _safe_eval(node):
    # 白名单 AST 遍历：只允许数字常量与基本算术运算符，其余一律拒绝
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _BINOPS:
        return _BINOPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARY:
        return _UNARY[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"不允许的表达式节点: {type(node).__name__}")

def calculator(expr: str) -> str:
    # 安全计算：'828 - 330' -> '498'
    try:
        return str(_safe_eval(ast.parse(expr.strip(), mode="eval")))
    except Exception as e:
        return f"ERROR: 无法计算 '{expr.strip()}'（{e}）"

# ---------- 工具 2: wiki_lookup —— 离线迷你百科（15 条词条） ----------
MINI_WIKI = {
    "eiffel tower":    "埃菲尔铁塔位于法国巴黎，为 1889 年世界博览会而建成，高 330 米。",
    "burj khalifa":    "哈利法塔位于阿联酋迪拜，2010 年落成，高 828 米，是世界最高建筑。",
    "mount everest":   "珠穆朗玛峰是地球最高峰，海拔 8849 米。",
    "moon":            "月球是地球唯一的天然卫星，与地球的平均距离约 384400 千米。",
    "light speed":     "真空中的光速约为每秒 299792 千米。",
    "nile river":      "尼罗河位于非洲，全长约 6650 千米。",
    "amazon river":    "亚马逊河位于南美洲，全长约 6400 千米，是流量最大的河流。",
    "great wall":      "中国长城各时期墙体总长约 21196 千米。",
    "pacific ocean":   "太平洋是地球最大的海洋，面积约 1.65 亿平方千米。",
    "marie curie":     "玛丽·居里（1867–1934），物理学家与化学家，1903、1911 年两次获诺贝尔奖。",
    "albert einstein": "阿尔伯特·爱因斯坦（1879–1955），1921 年获诺贝尔物理学奖。",
    "shakespeare":     "威廉·莎士比亚（1564–1616），英国剧作家与诗人。",
    "python":          "Python 编程语言由 Guido van Rossum 创造，首个版本发布于 1991 年。",
    "transformer":     "Transformer 架构由论文 Attention Is All You Need 提出，发表于 2017 年。",
    "react paper":     "ReAct 论文（Yao et al., arXiv:2210.03629）发表于 2022 年，提出推理与行动交错的 agent 范式。",
}

def wiki_lookup(query: str) -> str:
    q = query.strip().lower().strip("'").strip('"')
    if q in MINI_WIKI:
        return MINI_WIKI[q]
    for k, v in MINI_WIKI.items():   # 宽松兜底：子串互相包含也算命中
        if q in k or k in q:
            return v
    return (f"ERROR: 离线百科中没有词条 '{query.strip()}'。"
            f"可用词条: {', '.join(sorted(MINI_WIKI))}")

# ---------- 工具 3/4: note_write / note_read —— 便签 ----------
NOTEPAD = []

def note_write(text: str) -> str:
    NOTEPAD.append(text.strip())
    return f"已写入便签（第 {len(NOTEPAD)} 条）。"

def note_read(_: str = "") -> str:
    if not NOTEPAD:
        return "（便签为空）"
    return "\n".join(f"[{i+1}] {t}" for i, t in enumerate(NOTEPAD))

TOOLS = {
    "calculator": calculator,
    "wiki_lookup": wiki_lookup,
    "note_write": note_write,
    "note_read": note_read,
}

TOOL_DESCRIPTIONS = "\n".join([
    "- calculator: 安全算术求值，输入如 '828 - 330'",
    "- wiki_lookup: 查询离线迷你百科，输入英文词条名，如 'eiffel tower'",
    "- note_write: 把一条文本写入便签",
    "- note_read: 读取全部便签（输入留空即可）",
])

In [ ]:
# 工具集自检（全离线、确定性）
assert calculator("828 - 330") == "498"
assert calculator("__import__('os')").startswith("ERROR")   # 代码注入被白名单拒绝
assert "330" in wiki_lookup("eiffel tower")
assert wiki_lookup("golden gate bridge").startswith("ERROR")
print(note_write("测试便签"))
print(note_read())
NOTEPAD.clear()
print("✅ 工具集自检通过，共", len(MINI_WIKI), "条词条")

## 2 · ReAct system prompt：文法 + few-shot

ReAct 是一种**文本协议**（讲解第 2 节）：system prompt 约定行格式，让自由文本变得机器可解析。两个关键工程点：

- **stop sequence**：调 LLM 时设 `stop=["Observation:"]`——`Observation:` 行**永远由 harness 执行真实工具后回填，绝不让模型自己写**。这是防"幻觉观察"的第一道防线（讲解第 2 节 warn 框）；
- **few-shot**：放一个完整轨迹示例。对小模型（Qwen2.5-1.5B 级别）这几乎是必需的格式稳定器。

In [ ]:
REACT_SYSTEM_PROMPT = f'''你是一个严格遵循 ReAct 文法的助手。可用工具：
{TOOL_DESCRIPTIONS}

逐步解决问题。每一步只输出：
Thought: <你的推理：下一步做什么、为什么>
Action: <工具名，必须是 calculator / wiki_lookup / note_write / note_read 之一>
Action Input: <工具输入>

然后立即停止输出，等待系统回填 Observation。
Observation 由系统执行真实工具后写入，你绝不能自己编写 Observation。
当信息足够时，输出：
Thought: <总结推理>
Final Answer: <最终答案>

示例：
Question: 埃菲尔铁塔建成那一年加上 100 是哪一年？
Thought: 先查埃菲尔铁塔的建成年份。
Action: wiki_lookup
Action Input: eiffel tower
Observation: 埃菲尔铁塔位于法国巴黎，为 1889 年世界博览会而建成，高 330 米。
Thought: 建成于 1889 年，1889 + 100 用计算器确认。
Action: calculator
Action Input: 1889 + 100
Observation: 1989
Thought: 信息足够了。
Final Answer: 1989 年。

现在开始。'''

print(REACT_SYSTEM_PROMPT[:160] + "\n...（共", len(REACT_SYSTEM_PROMPT), "字符）")

## 3 · `parse_step`：regex 解析与格式漂移兜底

解析器是 ReAct 可靠性的咽喉（讲解第 3 节漂移表）。返回约定：

- 工具步 → `{"thought": str, "action": str, "input": str, "final": None}`
- 终答步 → `{"thought": str, "action": None, "input": None, "final": str}`
- 无法解析 → `None`（由主循环回喂格式提醒，**不是**直接判轨迹失败）

兜底设计：`re.IGNORECASE` 扛大小写漂移、`re.DOTALL` 支持多行 Action Input、解析前剥掉 `**`/反引号扛 Markdown 污染、缺 Action Input 判解析失败、编造工具名留给**执行层**拒绝（解析层不管语义）。

In [ ]:
FINAL_RE = re.compile(r"Final\s*Answer\s*[:：]\s*(?P<ans>.+)", re.IGNORECASE | re.DOTALL)
ACTION_RE = re.compile(
    r"Action\s*[:：]\s*(?P<action>[\w\.\-]+)\s*\n+\s*Action\s*Input\s*[:：]\s*(?P<input>.+)",
    re.IGNORECASE | re.DOTALL)
THOUGHT_RE = re.compile(r"Thought\s*[:：]\s*(?P<th>.+?)(?=\n\s*(?:Action|Final)|\Z)",
                        re.IGNORECASE | re.DOTALL)

def parse_step(text):
    '''解析 LLM 单步输出 -> dict(thought, action, input, final) 或 None。'''
    text = text.replace("**", "").replace("`", "")   # 剥 Markdown 污染
    thought_m = THOUGHT_RE.search(text)
    thought = thought_m.group("th").strip() if thought_m else ""
    final_m = FINAL_RE.search(text)
    if final_m:   # Final Answer 优先
        return {"thought": thought, "action": None, "input": None,
                "final": final_m.group("ans").strip()}
    action_m = ACTION_RE.search(text)
    if action_m:
        # 若模型把后续标签（如自己编的 Observation）也写进来了，截断之
        inp = re.split(r"\n\s*(?:Observation|Thought)\s*[:：]",
                       action_m.group("input"))[0].strip()
        return {"thought": thought, "action": action_m.group("action").strip(),
                "input": inp, "final": None}
    return None   # 彻底乱格式

print(parse_step("Thought: 查月球。\nAction: wiki_lookup\nAction Input: moon"))
print(parse_step("Thought: 够了。\nFinal Answer: 498 米"))
print(parse_step("一段完全没有标签的散文输出"))

## 4 · LLM 三级回退

统一接口：`llm(prompt, stop=None) -> str`。三个后端按开头检测结果自动选择：

1. **openai**：`gpt-4o-mini`，server 端原生支持 `stop`；
2. **local**：`Qwen/Qwen2.5-1.5B-Instruct`（⚠️ 首次运行下载 **约 3.1GB**；CPU 可跑，每步约几十秒；本地生成没有 server 端 stop，事后手工截断）；
3. **mock**（默认）：`ScriptedMockLLM` 按"prompt 中出现的最长 key"选剧本、按调用次数逐步回放——**确定性复现**教学轨迹，包括我们故意安排的死循环失败。

> 评测视角：mock LLM 就是 harness 的"测试替身"（test double）。先用它验证循环/解析/预算逻辑全对，再换真模型——否则你分不清是模型笨还是 harness 有 bug。

In [ ]:
class ScriptedMockLLM:
    '''按预置脚本回放的"模型"。
    scripts: {问题关键句: [第1步输出, 第2步输出, ...]}
    - 用 prompt 中出现的最长 key 选剧本（让"带反思重试"的剧本能覆盖原始剧本）；
    - 剧本耗尽则重复最后一步（用于确定性复现死循环失败模式）。'''
    def __init__(self, scripts):
        self.scripts = scripts
        self.counters = {}

    def __call__(self, prompt, stop=None):
        keys = [k for k in self.scripts if k in prompt]
        if not keys:
            return "Final Answer: （mock 剧本未覆盖此问题；要自由提问请用 openai/local 后端）"
        key = max(keys, key=len)
        i = self.counters.get(key, 0)
        self.counters[key] = i + 1
        steps = self.scripts[key]
        return steps[min(i, len(steps) - 1)]

MOCK_SCRIPTS = {
    # Q1：两次百科 + 一次计算
    "哈利法塔": [
        "Thought: 先查哈利法塔的高度。\nAction: wiki_lookup\nAction Input: burj khalifa",
        "Thought: 哈利法塔高 828 米，再查埃菲尔铁塔的高度。\nAction: wiki_lookup\nAction Input: eiffel tower",
        "Thought: 828 - 330 即为高度差，用计算器确认。\nAction: calculator\nAction Input: 828 - 330",
        "Thought: 高度差 498 米，信息足够。\nFinal Answer: 哈利法塔比埃菲尔铁塔高 498 米。",
    ],
    # Q2：两次百科 + 一次除法
    "光从月球": [
        "Thought: 先查月球到地球的距离。\nAction: wiki_lookup\nAction Input: moon",
        "Thought: 距离约 384400 千米，再查光速。\nAction: wiki_lookup\nAction Input: light speed",
        "Thought: 时间 = 距离 / 速度 = 384400 / 299792。\nAction: calculator\nAction Input: 384400 / 299792",
        "Thought: 约 1.28 秒，信息足够。\nFinal Answer: 光从月球到地球大约需要 1.28 秒。",
    ],
    # Q3：两次百科 + 计算 + 写便签（覆盖全部 4 个工具中的 3 个）
    "尼罗河": [
        "Thought: 先查尼罗河的长度。\nAction: wiki_lookup\nAction Input: nile river",
        "Thought: 尼罗河约 6650 千米，再查亚马逊河的长度。\nAction: wiki_lookup\nAction Input: amazon river",
        "Thought: 6650 - 6400 即为长度差。\nAction: calculator\nAction Input: 6650 - 6400",
        "Thought: 把结论写进便签。\nAction: note_write\nAction Input: 尼罗河比亚马逊河长 250 千米",
        "Thought: 已记录，信息足够。\nFinal Answer: 尼罗河比亚马逊河长约 250 千米（已写入便签）。",
    ],
    # 失败剧本：只有一步——会被无限重放，确定性复现 action loop + budget exceeded
    "金门大桥": [
        "Thought: 查金门大桥的主跨长度。\nAction: wiki_lookup\nAction Input: golden gate bridge",
    ],
    # Reflexion 重试剧本：key 取自反思文本（比 "金门大桥" 长，按最长 key 优先被选中）
    "重复同一查询不会有新结果": [
        "Thought: 反思指出离线百科没有金门大桥词条，重复查询无意义。先确认一次词条确实不存在。\nAction: wiki_lookup\nAction Input: golden gate bridge",
        "Thought: 确认词条不存在。遵循反思：不再重复查询，如实说明知识边界，避免编造数字。\nFinal Answer: 离线迷你百科中没有金门大桥词条，无法据此给出主跨长度；该问题超出本知识库范围。",
    ],
}

In [ ]:
if BACKEND == "openai":
    from openai import OpenAI
    _client = OpenAI()

    def llm(prompt, stop=None):
        resp = _client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0, max_tokens=400, stop=stop)
        return resp.choices[0].message.content

elif BACKEND == "local":
    # ⚠️ 首次运行下载 Qwen/Qwen2.5-1.5B-Instruct 约 3.1GB
    from transformers import AutoModelForCausalLM, AutoTokenizer
    _MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
    _tok = AutoTokenizer.from_pretrained(_MODEL_ID)
    _model = AutoModelForCausalLM.from_pretrained(_MODEL_ID, torch_dtype="auto",
                                                  device_map="auto")

    def llm(prompt, stop=None):
        text = _tok.apply_chat_template([{"role": "user", "content": prompt}],
                                        tokenize=False, add_generation_prompt=True)
        inputs = _tok(text, return_tensors="pt").to(_model.device)
        out = _model.generate(**inputs, max_new_tokens=300, do_sample=False,
                              pad_token_id=_tok.eos_token_id)
        gen = _tok.decode(out[0][inputs["input_ids"].shape[1]:],
                          skip_special_tokens=True)
        for s in (stop or []):   # 本地生成没有 server 端 stop，事后截断
            gen = gen.split(s)[0]
        return gen

else:
    llm = ScriptedMockLLM(MOCK_SCRIPTS)

print(f"llm 就绪（backend = {BACKEND}）")

## 5 · `react_agent` 主循环

不到 40 行，覆盖四件事：**调 LLM →解析 → 执行工具 → Observation 回填**，外加：

- `max_steps=8` 硬预算（讲解第 4 节 danger 框：永远先写预算再写循环）；
- 解析失败时把格式提醒当 Observation 回喂（消耗一步预算，防无限纠错）；
- 编造工具名由执行层拒绝，回喂可用工具表；
- 完整轨迹记录。轨迹每步：`{step, thought, action, input, observation, raw}`，其中 `action` 的两个特殊取值：`"<final>"`（给出终答）、`"<parse_error>"`（格式解析失败）——练习 2/3 的轨迹分析器依赖这个约定。

In [ ]:
def react_agent(question, llm, tools, max_steps=8, extra_context=""):
    '''ReAct 主循环。返回 dict(question, answer, status, steps, trajectory)。
    status: "success"（给出 Final Answer）| "budget_exceeded"（耗尽 max_steps）'''
    header = REACT_SYSTEM_PROMPT
    if extra_context:
        header += "\n\n" + extra_context
    transcript = f"\n\nQuestion: {question}\n"
    trajectory = []
    for t in range(1, max_steps + 1):
        out = llm(header + transcript, stop=["Observation:"])   # stop: 防幻觉观察
        parsed = parse_step(out)
        if parsed is None:   # 解析失败 ≠ 轨迹失败：错误当 Observation 回喂
            obs = ("ERROR: 输出格式无法解析。请严格按 'Thought:/Action:/Action Input:' "
                   "或 'Thought:/Final Answer:' 的行格式输出。可用工具: " + ", ".join(tools))
            trajectory.append({"step": t, "thought": "", "action": "<parse_error>",
                               "input": None, "observation": obs, "raw": out})
            transcript += out.strip() + f"\nObservation: {obs}\n"
            continue
        if parsed["final"] is not None:
            trajectory.append({"step": t, "thought": parsed["thought"], "action": "<final>",
                               "input": None, "observation": None, "raw": out})
            return {"question": question, "answer": parsed["final"],
                    "status": "success", "steps": t, "trajectory": trajectory}
        action, action_input = parsed["action"], parsed["input"]
        if action in tools:
            obs = tools[action](action_input)
        else:   # 编造工具名：解析层放过，执行层拒绝
            obs = f"ERROR: 未知工具 '{action}'。可用工具: " + ", ".join(tools)
        trajectory.append({"step": t, "thought": parsed["thought"], "action": action,
                           "input": action_input, "observation": obs, "raw": out})
        transcript += out.strip() + f"\nObservation: {obs}\n"
    return {"question": question, "answer": None,
            "status": "budget_exceeded", "steps": max_steps, "trajectory": trajectory}

def print_trajectory(result, max_obs_len=90):
    print("=" * 72)
    print(f"Question: {result['question']}")
    for s in result["trajectory"]:
        print(f"--- step {s['step']} ---")
        if s["thought"]:
            print(f"  Thought: {s['thought']}")
        if s["action"] == "<final>":
            continue
        print(f"  Action: {s['action']}  |  Input: {s['input']}")
        obs = str(s["observation"])
        if len(obs) > max_obs_len:
            obs = obs[:max_obs_len] + "…"
        print(f"  Observation: {obs}")
    print(f"=> status={result['status']}  steps={result['steps']}  answer={result['answer']}")

## 6 · 跑 3 道多跳问题

每道都需要**百科查询 + 计算的组合**（单跳答不出来）。mock 后端下轨迹完全确定；openai/local 后端下轨迹会变化——观察它们与剧本的差异本身就是有趣的练习。

In [ ]:
QUESTIONS = [
    "哈利法塔比埃菲尔铁塔高多少米？",
    "光从月球到地球大约需要多少秒？",
    "尼罗河比亚马逊河长多少千米？把结论写进便签。",
]

NOTEPAD.clear()
results = []
for q in QUESTIONS:
    r = react_agent(q, llm, TOOLS)
    results.append(r)
    print_trajectory(r)
    print()

print("便签内容:")
print(note_read())

## 7 · 失败分析：故意问百科里没有的问题

`golden gate bridge` 不在 15 条词条里。mock 剧本只有一步且被无限重放——**确定性复现**讲解第 5 节表格中的两种失败模式：

- **action loop（死循环）**：连续多步行动指纹 `(action, input)` 相同；
- **budget exceeded**：耗尽 `max_steps` 仍无 Final Answer。

> 真实后端（openai/local）跑这道题未必死循环——模型可能读了 ERROR 里的可用词条表后直接承认知识边界，那是**更好**的行为。失败模式依赖于模型 × harness 的组合，这正是 [Kapoor 2024] 要求评测报告 harness 细节的原因。

In [ ]:
fail_q = "金门大桥的主跨有多长？"
fail_result = react_agent(fail_q, llm, TOOLS)
print_trajectory(fail_result)

In [ ]:
# 自动失败分类（对应讲解第 5 节表格的"检测信号"列，全部是启发式，有误报可能）
traj = fail_result["trajectory"]
fps = [(s["action"], s["input"]) for s in traj]

signals = []
if fail_result["status"] == "budget_exceeded":
    signals.append("budget exceeded：耗尽步数预算仍无 Final Answer")
for i in range(len(fps) - 2):
    if len(set(fps[i:i + 3])) == 1:
        signals.append(f"action loop（死循环）：连续 3 步行动指纹相同 {fps[i]}")
        break
n_tool_calls = sum(1 for s in traj if s["action"] in TOOLS)
if fail_result["status"] == "success" and n_tool_calls == 0:
    signals.append("premature answer 嫌疑：零工具调用即作答（ungrounded）")
made_up = {s["action"] for s in traj
           if s["action"] not in TOOLS and s["action"] not in ("<final>", "<parse_error>")}
if made_up:
    signals.append(f"tool misuse：编造工具名 {made_up}")
n_parse_fail = sum(1 for s in traj if s["action"] == "<parse_error>")
if n_parse_fail:
    signals.append(f"format failure：解析失败 {n_parse_fail} 次（报告时须与 task failure 区分）")

print("检测到的失败信号:")
for s in (signals or ["（无——任务成功，或失败模式未被以上启发式覆盖）"]):
    print(" -", s)

## 8 · Reflexion：失败后的自我反思重试

按讲解第 6 节：评估器判失败 → 反思 LM 读轨迹生成**语言化反思** → 把反思注入下一次尝试的上下文。参数零更新，全部"学习"发生在 in-context。

本实验的成功判据 `grounded_success`：给出 Final Answer **且**至少调用过一次工具（接地要求——对"百科里没有"的问题，接地的正确行为是查证后明确说明知识边界，而不是死循环、也不是编造数字）。

> ⚠️ 协议提醒：Reflexion 重试改变了评测协议（1 次尝试 → k 次尝试）。报告成绩必须注明 attempts 数（τ-bench 的 pass^k 思想 [Yao 2024]）。

In [ ]:
CANNED_REFLECTION = (
    "上次尝试失败：我反复用 wiki_lookup 查询 'golden gate bridge'，但离线百科没有该词条，"
    "重复同一查询不会有新结果，最终耗尽步数预算。下次应当：最多确认一次词条是否存在；"
    "若知识库确实没有，就直接在 Final Answer 中说明知识边界，而不是继续循环或编造数字。")

def reflect(question, result, llm):
    # mock 后端用预置反思保证可复现；真实后端让 LLM 读失败轨迹自己写
    if BACKEND == "mock":
        return CANNED_REFLECTION
    lines = [f"step{s['step']}: Action={s['action']}, Input={s['input']}, "
             f"Obs={str(s['observation'])[:60]}" for s in result["trajectory"]]
    prompt = ("以下 agent 轨迹未能在步数预算内回答问题。\n"
              f"问题: {question}\n轨迹:\n" + "\n".join(lines) +
              "\n\n请用 2-3 句话总结失败原因，并给出下次尝试的具体改进策略。")
    return llm(prompt, stop=None)

reflection = reflect(fail_q, fail_result, llm)
print("反思文本:\n" + reflection)

In [ ]:
# 带反思重试：反思文本注入 prompt（讲解第 6 节的 episodic memory 注入）
retry_result = react_agent(fail_q, llm, TOOLS,
                           extra_context="[过往反思——来自上一次失败的尝试]\n" + reflection)
print_trajectory(retry_result)

def grounded_success(result):
    n_tools = sum(1 for s in result["trajectory"] if s["action"] in TOOLS)
    return result["status"] == "success" and n_tools >= 1

print()
print("对比（同一问题，attempts=2）:")
for name, r in [("attempt 1（无反思）", fail_result), ("attempt 2（带反思）", retry_result)]:
    print(f"  {name}: status={r['status']}, steps={r['steps']}, "
          f"grounded_success={grounded_success(r)}")
print("\n⚠️ 报告 Reflexion 成绩时必须注明 attempts 数（pass^k），否则与单次尝试不可比。")

## ✏️ 练习 1：从零重写 `parse_step`

不回看第 3 节的实现，自己写一个 `parse_step_v2(text)`，返回约定与 `parse_step` 完全相同：

- 工具步 → `{"thought": str, "action": str, "input": str, "final": None}`
- 终答步 → `{"thought": str, "action": None, "input": None, "final": str}`
- 无法解析 → `None`

需要扛住的 6 类输入（即下方自测用例）：①标准格式；②多行 Action Input；③大小写/空白漂移；④缺 Action Input（→ `None`）；⑤直接 Final Answer；⑥彻底乱格式（→ `None`）。

**提示**：`re.IGNORECASE | re.DOTALL`；先判 Final Answer 再判 Action；缺 Action Input 时整体判解析失败。10–20 行可完成。

In [ ]:
def parse_step_v2(text):
    # TODO 1: 尝试匹配 Final Answer（IGNORECASE；命中则提取 thought 一起返回）
    # TODO 2: 尝试匹配 Action + Action Input（DOTALL 让 input 可跨多行）
    # TODO 3: 提取 Thought（可为空字符串 ""）
    # TODO 4: 都不匹配 -> return None
    raise NotImplementedError("完成上面的 TODO 后删除此行")

In [ ]:
# ---- 练习 1 自测（6 个用例） ----
r1 = parse_step_v2("Thought: 先查月球距离。\nAction: wiki_lookup\nAction Input: moon")
assert r1 is not None and r1["final"] is None, "用例1: 标准格式应解析为工具步"
assert r1["action"] == "wiki_lookup" and r1["input"] == "moon"
assert "月球" in r1["thought"]

r2 = parse_step_v2("Thought: 记一条多行便签。\nAction: note_write\nAction Input: 第一行\n第二行")
assert r2 is not None and r2["action"] == "note_write", "用例2: 多行 input"
assert "第一行" in r2["input"] and "第二行" in r2["input"]

r3 = parse_step_v2("thought: 查一下。\naction :  wiki_lookup\naction input: moon")
assert r3 is not None and r3["action"] == "wiki_lookup" and r3["input"] == "moon", \
    "用例3: 大小写/空白漂移也要能解析"

assert parse_step_v2("Thought: 想用计算器。\nAction: calculator") is None, \
    "用例4: 缺 Action Input 应判解析失败"

r5 = parse_step_v2("Thought: 信息足够。\nFinal Answer: 498 米")
assert r5 is not None and r5["final"] == "498 米" and r5["action"] is None, "用例5: 终答步"

assert parse_step_v2("我觉得应该先查点资料，然后再算一算。") is None, "用例6: 乱格式返回 None"

print("✅ 练习 1 通过")

## ✏️ 练习 2：`detect_loop` —— 死循环检测器

实现讲解第 4 节的行动指纹检测：`detect_loop(trajectory, window=3)`。

- 每步指纹 = `(action, input)` 二元组；
- 轨迹中存在**任意连续 `window` 步**指纹完全相同 → 返回 `True`，否则 `False`；
- 轨迹长度不足 `window` → `False`。

**提示**：滑动窗口 + `set` 去重判断，5–8 行可完成。注意：同一 action 配不同 input **不算**循环（那可能是合理的逐项查询）。

In [ ]:
def detect_loop(trajectory, window=3):
    # trajectory: list[dict]，每步至少含 "action" 与 "input" 两个键
    # TODO 1: 提取指纹序列 fps = [(action, input), ...]
    # TODO 2: 滑动窗口检查任意连续 window 个指纹是否完全相同
    raise NotImplementedError("完成上面的 TODO 后删除此行")

In [ ]:
# ---- 练习 2 自测 ----
def _mk(steps):   # 合成轨迹: [(action, input), ...] -> list[dict]
    return [{"action": a, "input": i} for a, i in steps]

t_loop = _mk([("wiki_lookup", "moon"),
              ("wiki_lookup", "golden gate bridge"),
              ("wiki_lookup", "golden gate bridge"),
              ("wiki_lookup", "golden gate bridge")])
assert detect_loop(t_loop), "中段连续 3 步相同指纹应判定为循环"

t_ok = _mk([("wiki_lookup", "moon"), ("wiki_lookup", "light speed"),
            ("calculator", "384400 / 299792"), ("<final>", None)])
assert not detect_loop(t_ok), "正常轨迹不应误报"

assert not detect_loop(_mk([("wiki_lookup", "x"), ("wiki_lookup", "x")])), \
    "长度不足 window 应返回 False"
assert detect_loop(_mk([("wiki_lookup", "x"), ("wiki_lookup", "x")]), window=2), \
    "window=2 时两步相同即判循环"
assert not detect_loop(_mk([("wiki_lookup", "a"), ("wiki_lookup", "b"), ("wiki_lookup", "c")])), \
    "同 action 不同 input 不算循环"
assert not detect_loop([]), "空轨迹返回 False"

if BACKEND == "mock":   # mock 后端下第 7 节的失败轨迹是确定性死循环
    assert detect_loop(fail_result["trajectory"]), "应能在真实失败轨迹上检出循环"

print("✅ 练习 2 通过")

## ✏️ 练习 3：`step_budget_report` —— 轨迹预算报告

实现讲解第 1 节预告的"轨迹级指标"最小版：`step_budget_report(trajectory, max_steps=8)` 返回：

```python
{"n_steps": int,            # 轨迹总步数
 "tool_counts": dict,       # 工具调用分布，如 {"wiki_lookup": 2, "calculator": 1}
 "budget_exceeded": bool}   # 是否超步
```

约定（与 `react_agent` 的轨迹 schema 一致）：`action == "<final>"` 是终答步、`"<parse_error>"` 是解析失败步，**二者都不计入** `tool_counts`；`budget_exceeded` = 轨迹中没有 `"<final>"` 步**且** `n_steps >= max_steps`。

**提示**：`collections.Counter`，5–8 行可完成。

In [ ]:
def step_budget_report(trajectory, max_steps=8):
    # TODO 1: n_steps = 轨迹长度
    # TODO 2: tool_counts —— 用 Counter 统计 action 分布，排除 "<final>" / "<parse_error>" / None
    # TODO 3: budget_exceeded —— 无 "<final>" 步 且 n_steps >= max_steps
    raise NotImplementedError("完成上面的 TODO 后删除此行")

In [ ]:
# ---- 练习 3 自测 ----
traj_ok = [
    {"action": "wiki_lookup", "input": "burj khalifa"},
    {"action": "wiki_lookup", "input": "eiffel tower"},
    {"action": "calculator", "input": "828 - 330"},
    {"action": "<final>", "input": None},
]
rep = step_budget_report(traj_ok, max_steps=8)
assert rep["n_steps"] == 4
assert rep["tool_counts"] == {"wiki_lookup": 2, "calculator": 1}
assert rep["budget_exceeded"] == False, "成功轨迹不算超步"

traj_bad = [{"action": "wiki_lookup", "input": "golden gate bridge"}] * 8
rep2 = step_budget_report(traj_bad, max_steps=8)
assert rep2["n_steps"] == 8
assert rep2["tool_counts"] == {"wiki_lookup": 8}
assert rep2["budget_exceeded"] == True, "8 步无 <final> 应判超步"

traj_mix = [{"action": "<parse_error>", "input": None},
            {"action": "calculator", "input": "1+1"},
            {"action": "<final>", "input": None}]
rep3 = step_budget_report(traj_mix, max_steps=8)
assert rep3["tool_counts"] == {"calculator": 1}, "解析失败步不计入工具分布"
assert rep3["budget_exceeded"] == False

print("✅ 练习 3 通过")

## 📖 参考答案

> **先自己做，再对照。** 每题一个独立 cell；运行参考实现会覆盖你的版本，之后可回去重跑对应的自测 cell 验证。

In [ ]:
# 练习 1 参考答案 —— 先自己做，再对照
def parse_step_v2(text):
    text = text.replace("**", "").replace("`", "")   # 剥 Markdown 污染
    th_m = re.search(r"Thought\s*[:：]\s*(?P<th>.+?)(?=\n\s*(?:Action|Final)|\Z)",
                     text, re.IGNORECASE | re.DOTALL)
    thought = th_m.group("th").strip() if th_m else ""
    final_m = re.search(r"Final\s*Answer\s*[:：]\s*(?P<ans>.+)",
                        text, re.IGNORECASE | re.DOTALL)
    if final_m:   # Final Answer 优先
        return {"thought": thought, "action": None, "input": None,
                "final": final_m.group("ans").strip()}
    action_m = re.search(
        r"Action\s*[:：]\s*(?P<action>[\w\.\-]+)\s*\n+\s*Action\s*Input\s*[:：]\s*(?P<input>.+)",
        text, re.IGNORECASE | re.DOTALL)
    if action_m:
        inp = re.split(r"\n\s*(?:Observation|Thought)\s*[:：]",
                       action_m.group("input"))[0].strip()
        return {"thought": thought, "action": action_m.group("action").strip(),
                "input": inp, "final": None}
    return None   # 缺 Action Input / 彻底乱格式

print("参考答案 1 已载入，可回去重跑练习 1 自测 cell")

In [ ]:
# 练习 2 参考答案 —— 先自己做，再对照
def detect_loop(trajectory, window=3):
    fps = [(s["action"], s["input"]) for s in trajectory]
    for i in range(len(fps) - window + 1):
        if len(set(fps[i:i + window])) == 1:   # 窗口内指纹去重后只剩 1 种
            return True
    return False

print("参考答案 2 已载入，可回去重跑练习 2 自测 cell")

In [ ]:
# 练习 3 参考答案 —— 先自己做，再对照
def step_budget_report(trajectory, max_steps=8):
    finished = any(s["action"] == "<final>" for s in trajectory)
    tool_counts = Counter(s["action"] for s in trajectory
                          if s["action"] not in (None, "<final>", "<parse_error>"))
    return {"n_steps": len(trajectory),
            "tool_counts": dict(tool_counts),
            "budget_exceeded": (not finished) and len(trajectory) >= max_steps}

print("参考答案 3 已载入，可回去重跑练习 3 自测 cell")

## 小结

- **agent = while 循环里的 LLM**：`react_agent` 不到 40 行，覆盖调用、解析、执行、回填、预算五件事——框架只是这个循环的包装；
- **Observation 永远由 harness 回填**（stop sequence 截断），这是防幻觉观察的第一道防线；
- **解析失败 ≠ 轨迹失败**：格式提醒回喂给模型自我修复的机会，但消耗预算；评测报告中 format failure 必须与 task failure 分开统计；
- **评测对象从"单个回答"变成"整条轨迹"**：步数、工具调用分布、失败模式分类成为一等公民指标——练习 2/3 就是你的第一个轨迹分析器；
- **Reflexion** 用语言反馈替代梯度做 in-context 策略改进；报告成绩必须注明 attempts 数（pass^k）。

**下一步 →** 模块 03《沙箱、轨迹与 agent harness》（`../03_sandbox_harness/03_讲解.html`）：把本章手搓的循环系统化为可复现、可隔离、可规模化的评测 harness。

---
## 🎯 真实数据胶囊题：真实 GSM8K 步骤上的 ReAct 工具循环

ReAct = 交替 Thought / Action(调工具) / Observation。GSM8K 答案的 `<<...>>` 是一串真实的计算 action。实现一个 ReAct 执行器：按顺序执行真实算式步骤，最终结果应等于金标答案。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.ai_agents_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn,headers=None):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p):
        req=urllib.request.Request(url, headers=headers or {})
        open(p,"wb").write(urllib.request.urlopen(req,timeout=40).read())
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def mbpp(n=100):
    p=_f("https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl","mbpp.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]

rows=gsm8k(100)
def calc(expr):
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expr): raise ValueError
    return float(eval(expr,{"__builtins__":{}},{}))
def steps_of(ans): return re.findall(r"<<([^=]+)=([^>]+)>>", ans)
print("示例真实 action 序列:", steps_of(rows[0]["answer"]))

**练习**：实现 `react_run(answer)`：依次对每个 `<<lhs=rhs>>` 调 `calc(lhs)`，返回**最后一步**的计算结果（模拟 agent 算到的最终数）。

In [ ]:
def react_run(answer):
    # TODO: 取 steps_of(answer)，依次 calc(lhs)，返回最后一步结果(float)；无步骤返回 None
    raise NotImplementedError


In [ ]:
# 自测：最后一步结果应等于金标答案
match=tot=0
for r in rows[:60]:
    res=react_run(r["answer"])
    if res is None: continue
    tot+=1
    try: match += abs(res - float(gold(r["answer"]))) < 1e-6
    except: pass
assert tot>0 and match/tot > 0.8, f"ReAct 末步应≈金标>80%, 得到{match}/{tot}"
print(f"ReAct 工具循环末步 == 金标答案: {match}/{tot} ✓")


### 📖 参考答案

In [ ]:
def react_run(answer):
    steps=steps_of(answer)
    if not steps: return None
    last=None
    for lhs,rhs in steps: last=calc(lhs)
    return last
print("✓ ReAct 把推理拆成可执行的 action 序列，工具补足模型不擅长的精确计算")